# 04: Contrastive Learning for Batch Detection

In mass spectrometry, batch effects (variations in instrument sensitivity over time) can often mask true biological signals. This notebook demonstrates how **Contrastive Learning (SimCLR)** with a **Transformer encoder** can learn robust embeddings that prioritize class-relevant features while being invariant to batch-level variance.

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"
import pandas as pd, numpy as np, torch, torch.nn.functional as F
import plotly.express as px, plotly.graph_objects as go
from sklearn.manifold import TSNE
from fishy.experiments.contrastive import ContrastiveConfig, run_contrastive_experiment
from fishy.data.module import create_data_module
from fishy._core.utils import get_device

# 1. Configure SimCLR with a Transformer encoder
# We train for 100 epochs to allow the self-supervised representation to stabilize.
config = ContrastiveConfig(
    contrastive_method="simclr", 
    encoder_type="transformer", 
    dataset="batch-detection", 
    num_epochs=100,
    batch_size=32,
    embedding_dim=64
)

print("Running SimCLR training...")
results = run_contrastive_experiment(config)

## 1. Pair-wise Performance

In contrastive learning, the goal is to maximize the similarity between positive pairs (same class) and minimize it for negative pairs (different classes). This plot shows the accuracy of correctly classifying a pair as "Same" or "Different" based on their cosine similarity in the learned embedding space.

In [ ]:
history = results.get("history")
if history:
    px.line(y=history["accuracy"], title="Pair-wise Prediction Accuracy over 100 Epochs", 
            labels={'x':'Epoch', 'y':'Accuracy'}, template="plotly_white")

## 2. Embedding Visualization (t-SNE)

We project the high-dimensional embeddings produced by the Transformer encoder into a 2D space. If the contrastive learning was successful, samples from the same class or batch should cluster together, demonstrating that the model has learned meaningful spectral features.

In [ ]:
# We extract the embeddings from the trained model
print("Generating t-SNE projection of the embedding space...")

# For this example, we assume we have the original data labels
dm = create_data_module("batch-detection")
dm.setup()
X, _ = dm.get_numpy_data()
names = dm.get_class_names()
y_indices = np.argmax(_, axis=1) if _.ndim > 1 else _.flatten().astype(int)

model = results.get("model")
if model:
    model.eval()
    with torch.no_grad():
        # Transformer expects 3D input (B, L, 1)
        X_t = torch.tensor(X).unsqueeze(-1).to(get_device())
        
        # Use the new forward_one method for robust embedding extraction
        if hasattr(model, "forward_one"):
            embeddings = model.forward_one(X_t).cpu().numpy()
        else:
            # Fallback to direct encoder access
            encoder = getattr(model, "encoder", getattr(model, "backbone", model))
            embeddings = encoder(X_t).cpu().numpy()
    
    tsne = TSNE(n_components=2, random_state=42)
    X_emb_2d = tsne.fit_transform(embeddings)
    
    px.scatter(x=X_emb_2d[:,0], y=X_emb_2d[:,1], color=[names[i] for i in y_indices],
               title="t-SNE of Transformer Embeddings", template="plotly_white").show()


## 3. Cosine Similarity Distribution

We compare the distribution of cosine similarities for positive pairs (same class) versus negative pairs. An ideal model will have two distinct, non-overlapping peaks: one near 1.0 for same-class samples and one near 0.0 (or lower) for different-class samples.

In [ ]:
# This logic is typically computed inside the evaluate_pairwise_performance method of ContrastiveTrainer
print(f"Final Pair-wise F1 Score: {results.get('val_f1', 0):.4f}")
print(f"Final Pair-wise Balanced Accuracy: {results.get('val_balanced_accuracy', 0):.4f}")